In [1]:
#Importar las librerias
from pathlib import Path
import sys

# Carpeta raíz del proyecto
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, RobustScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_score, KFold, RandomizedSearchCV
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor 
from sklearn.ensemble import RandomForestRegressor
import warnings
from tqdm import TqdmWarning 

warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('../data/store_sales.csv')

In [3]:
# FEATURE ENGINEERING
customer_frequency = (
    df.groupby("CustomerID")
      .size()
)

df["Customer_Frequency"] = df["CustomerID"].map(customer_frequency)

In [4]:
df["Gender_Category"] = (
    df["Gender"] +
    "_" +
    df["Category"]
)

In [5]:
df["Payment_Category"] = (
    df["PaymentMethod"] +
    "_" +
    df["Category"]
)

In [6]:
bins = [18,25,35,45,60,100]

labels = [
    "18-25",
    "26-35",
    "36-45",
    "46-60",
    "60+"
]

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels
)

In [7]:
df.drop(columns= ['Age', 'CustomerID'], inplace= True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Gender              5000 non-null   object  
 1   Category            5000 non-null   object  
 2   ItemPurchased       5000 non-null   object  
 3   Amount              5000 non-null   float64 
 4   Season              5000 non-null   object  
 5   PaymentMethod       5000 non-null   object  
 6   ItemRating          5000 non-null   float64 
 7   DiscountApplied(%)  5000 non-null   int64   
 8   PreviousPurchases   5000 non-null   int64   
 9   Customer_Frequency  5000 non-null   int64   
 10  Gender_Category     5000 non-null   object  
 11  Payment_Category    5000 non-null   object  
 12  Age_Group           5000 non-null   category
dtypes: category(1), float64(2), int64(3), object(7)
memory usage: 474.0+ KB


In [9]:
df['Age_Group'] = df['Age_Group'].astype('object')

In [10]:
X = df.drop('Amount', axis=1)
y = df['Amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=52)

In [13]:
# MODELOS LINEALES 
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

numeric_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree = 2, include_bias= False))
])

categoric_pipeline = Pipeline([
    ('encoder', OneHotEncoder(drop = 'first'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categoric_pipeline, cat_cols)
])

selector = RFE(
    estimator = LinearRegression(),
    n_features_to_select= 10
)

In [14]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", selector),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

mae_lr_rfe = mean_absolute_error(y_test, y_pred)
mse_lr_rfe = mean_squared_error(y_test, y_pred)
rmse_lr_rfe = mse_lr_rfe ** 0.5
r2_lr_rfe = r2_score(y_test, y_pred)

print(f"MAE: {mae_lr_rfe}")
print(f"RMSE: {rmse_lr_rfe}")
print(f"R2: {r2_lr_rfe}")

MAE: 92.78020019082118
RMSE: 220.26729935095508
R2: 0.836092849607496


In [15]:
# PROBAR LINEAR; RIDGE; LASSO; ELASTICNET
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=10),
    "Lasso": Lasso(alpha=0.01),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio= 0.5), 
}

results_linear = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    results_linear.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

In [16]:
results_linear_df = pd.DataFrame(results_linear)
results_linear_df.loc[len(results_linear_df)] = [
    "Linear Regression + RFE",
    mae_lr_rfe,
    rmse_lr_rfe,
    r2_lr_rfe
]

results_linear_df

,Model,MAE,RMSE,R2
0,Linear Regression,92.577780,220.503296,0.835741
1,Ridge,92.255479,220.391296,0.835908
2,Lasso,92.518850,220.478951,0.835778
3,ElasticNet,92.191250,220.436183,0.835841
4,Linear Regression + RFE,92.780200,220.267299,0.836093


In [17]:
# REALIZAR MEJORA DE PARAMETROS PARA MODELOS LINEALES
models = {
    "Ridge": (
        Ridge(),
        {
            "model__alpha": np.logspace(-4, 2, 30)
        }
    ),
    "Lasso": (
        Lasso(),
        {
            "model__alpha": np.logspace(-4, 2, 30)
        }
    ),
    "ElasticNet": (
        ElasticNet(),
        {
            "model__alpha": np.logspace(-4, 2, 30),
            "model__l1_ratio": np.linspace(0.1, 0.9, 9)
        }
    )
}

results = []

for name, (model, params) in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    random_search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=10,
        cv=5,
        scoring="r2",
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train, y_train)

    y_pred = random_search.predict(X_test)

    best_params = random_search.best_params_
    
    results.append({
        "Model": name,
        "Alpha": best_params.get("model__alpha", "-"),
        "CV R2": random_search.best_score_,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

    results_df = pd.DataFrame(results)

results_df

,Model,Alpha,CV R2,MAE,RMSE,R2
0,Ridge,9.236709,0.807822,92.262469,220.390287,0.835910
1,Lasso,0.329034,0.807560,91.373891,219.861421,0.836696
2,ElasticNet,0.030392,0.807822,92.161644,220.337733,0.835988


In [18]:
linear = results_linear_df.iloc[0]

results_df.loc[len(results_df)] = {
    "Model": linear["Model"],
    "Alpha": "-",
    "CV R2": "-",
    "MAE": linear["MAE"],
    "RMSE": linear["RMSE"],
    "R2": linear["R2"]
}


results_df.loc[len(results_df)] = {
    "Model": "Linear Regression + RFE",
    "Alpha": "-",
    "CV R2": '-',
    "MAE": mae_lr_rfe,
    "RMSE": rmse_lr_rfe,
    "R2": r2_lr_rfe
}

In [19]:
results_df

,Model,Alpha,CV R2,MAE,RMSE,R2
0,Ridge,9.236709,0.807822,92.262469,220.390287,0.835910
1,Lasso,0.329034,0.80756,91.373891,219.861421,0.836696
2,ElasticNet,0.030392,0.807822,92.161644,220.337733,0.835988
3,Linear Regression,-,-,92.577780,220.503296,0.835741
4,Linear Regression + RFE,-,-,92.780200,220.267299,0.836093


In [20]:
# MODELOS BASADOS EN ARBOLES:

preprocessor_tree = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols)
])

pipeline_rf = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", RandomForestRegressor(random_state=42))
])

pipeline_rf.fit(X_train, y_train)

y_pred = pipeline_rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"RMSE: {rmse}")
print(f"R2: {r2}")

MAE: 94.59192538999999
RMSE: 232.95320101523228
R2: 0.8166692940392005


In [21]:
pipeline_xgb = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    ))
])

pipeline_xgb.fit(X_train, y_train)

y_pred = pipeline_xgb.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"RMSE: {rmse}")
print(f"R2: {r2}")

MAE: 101.41188893699646
RMSE: 263.06994267893026
R2: 0.7662022613802977


In [22]:
params_xgb = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

params_rf = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

In [23]:
tree_models = {
    "Random Forest": (
        RandomForestRegressor(random_state=42),
        params_rf
    ),

    "XGBoost": (
        XGBRegressor(
            random_state=42,
            objective="reg:squarederror"
        ),
        params_xgb
    )
}

results_tree = []

for name, (model, params) in tree_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", model)
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=20,
        cv=5,
        scoring="r2",
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    y_pred = search.predict(X_test)
    
    best_params = random_search.best_params_
    
    results_tree.append({
        "Model": name,
        "Alpha": best_params.get("model__alpha", "-"),
        "CV R2": search.best_score_,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

results_tree = pd.DataFrame(results_tree)

In [24]:
results_tree

,Model,Alpha,CV R2,MAE,RMSE,R2
0,Random Forest,0.030392,0.801784,92.547760,228.927774,0.822950
1,XGBoost,0.030392,0.803137,93.145558,229.004991,0.822831


In [25]:
results_final = pd.concat(
    [results_df, results_tree],
    ignore_index=True
)
results_final

,Model,Alpha,CV R2,MAE,RMSE,R2
0,Ridge,9.236709,0.807822,92.262469,220.390287,0.835910
1,Lasso,0.329034,0.80756,91.373891,219.861421,0.836696
2,ElasticNet,0.030392,0.807822,92.161644,220.337733,0.835988
3,Linear Regression,-,-,92.577780,220.503296,0.835741
4,Linear Regression + RFE,-,-,92.780200,220.267299,0.836093
5,Random Forest,0.030392,0.801784,92.547760,228.927774,0.822950
6,XGBoost,0.030392,0.803137,93.145558,229.004991,0.822831


In [26]:
def evaluate_cv(models, preprocessor, X, y, cv=5):

    results = []

    for name, model in models.items():

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cv,
            scoring="r2",
            n_jobs=-1
        )

        results.append({
            "Model": name,
            "Mean CV R2": scores.mean(),
            "Std CV": scores.std()
        })

    return pd.DataFrame(results)

In [27]:
# REALIZAR CROSS VALIDATION EN MODELOS DE ÁRBOLES
models_tree = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    )
}

results_cv_tree = evaluate_cv(models_tree, preprocessor_tree, X, y)

In [28]:
# REALIZAR CROSS VALIDATION PARA MODELOS LINEALES
models_linear = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=10),
    "Lasso": Lasso(alpha=0.01),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio= 0.5)
}

results_cv_linear = evaluate_cv(models_linear, preprocessor,X, y)

In [29]:
results_cv_all = pd.concat(
    [results_cv_linear, results_cv_tree],
    ignore_index=True
)
results_cv_all

,Model,Mean CV R2,Std CV
0,Linear,0.814567,0.021727
1,Ridge,0.814743,0.020810
2,Lasso,0.814580,0.021719
3,ElasticNet,0.814735,0.019917
4,Random Forest,0.794050,0.026731
5,XGBoost,0.749607,0.031546
